# 02 — Challenge Outputs (US2)

Run burn centers per 100k, rural/urban travel burden, pediatric access, burn-bed capacity; tables and visualizations.

In [18]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

from pipeline.ingest import ingest_nird
from pipeline.augment import build_analytic_table
from pipeline.bei_composite import (
    burn_centers_per_100k,
    rural_urban_travel_burden,
    pediatric_access_per_capita,
    burn_beds_per_100k,
    state_fips_to_abbr,
)
from pipeline.geocode import load_batch_results_and_merge
from pipeline import config

nird_path = config.NIRD_FULL_PATH if config.NIRD_FULL_PATH.exists() else config.NIRD_SAMPLE_PATH
facilities, _ = ingest_nird(nird_path)
# Use facility coordinates if GeocodeResults from 01_data_exploration exists (more accurate travel burden)
geocode_path = config.TABLES_DIR / "GeocodeResults.csv"
if geocode_path.exists():
    facilities = load_batch_results_and_merge(facilities, results_path=geocode_path)
try:
    tracts = build_analytic_table(facilities)
except Exception as e:
    print("Need TIGER/ACS/RUCA:", e)
    tracts = None

if tracts is not None and len(facilities) > 0:
    c100 = burn_centers_per_100k(facilities, tracts)
    print("Burn centers per 100k (state):")
    display(state_fips_to_abbr(c100).head(10))
    burden = rural_urban_travel_burden(tracts, facilities)
    print("Rural vs urban burden:")
    display(burden)
    peds = pediatric_access_per_capita(facilities, tracts)
    print("Pediatric access:")
    display(state_fips_to_abbr(peds).head())
    beds = burn_beds_per_100k(facilities, tracts)
    print("Beds per 100k:")
    display(state_fips_to_abbr(beds).head())

Ingest:   0%|          | 0/4 [00:00<?, ?step/s]

Augment:   0%|          | 0/5 [00:00<?, ?step/s]

Augment TIGER:   0%|          | 0/51 [00:00<?, ?shp/s]

Burn centers per 100k (state):


,state,pop,supply_weight,centers_per_100k
0,AL,5028092,2.10,0.041765
1,AK,734821,0.40,0.054435
2,AZ,7172282,3.70,0.051587
3,AR,3018669,1.50,0.049691
4,CA,39356104,21.50,0.054629
5,CO,5770790,5.80,0.100506
6,CT,3611317,3.00,0.083072
7,DE,993635,0.20,0.020128
8,DC,670587,2.25,0.335527
9,FL,21634529,11.25,0.052000


...

,is_rural,median,mean,count
0,False,42.507537,94.073667,68010
1,True,199.185772,251.551626,16405


Pediatric access:


,state,child_pop,peds_facilities,peds_per_100k_children
0,AL,1110642,1.0,0.090038
1,AK,179338,2.0,1.115213
2,AZ,1593463,4.0,0.251026
3,AR,697268,1.0,0.143417
4,CA,8774570,20.0,0.227931


Beds per 100k:


,state,pop,beds,beds_per_100k
0,AL,5028092,52.0,1.034190
1,AK,734821,0.0,0.000000
2,AZ,7172282,80.0,1.115405
3,AR,3018669,10.0,0.331272
4,CA,39356104,140.0,0.355726


Distance sanity check (air & ground via Valhalla)

**Check 1 — Mankato Census Tract 1716, Blue Earth, MN → Minneapolis, MN** (Hennepin Healthcare)

- **Origin tract**: `27013171600` — Mankato Census Tract 1716, Blue Earth, MN
- **Minneapolis**: 44.979970, -93.263840 (ZIP 55415, Hennepin Healthcare)

- Expected: Air 67.55 mi (108.71 km); Driving 82.42 mi (132.64 km), ~108 min

**Check 2 — Port Isabel, TX → Valley Baptist Medical Center–Harlingen** (AHA 6741790)
- **Port Isabel**: 26.073412, -97.208584 (Cameron County, TX; GNIS 4858892)
- **Valley Baptist–Harlingen**: 26.255523, -97.667502 (2101 Pease St, Harlingen, TX 78550)
- Expected: Air 31.12 mi (50.08 km); Driving 45.34 mi (72.96 km), ~57 min

## Processed travel-time matrices

Load the postprocessed travel-time matrices that were already generated.  
These files include filled durations and live in `Data/output/Travel Dist Processed`.

In [21]:
import numpy as np
import pandas as pd
from pipeline import config

assert tracts is not None, "Run Cell 1 first to build tract origins for name lookups."
assert len(facilities) > 0, "Run Cell 1 first to load facilities for name lookups."

processed_dir = config.OUTPUT_DIR / "Travel Dist Processed"
processed_paths = {
    "mn_high_detail": processed_dir / "valhalla_mn_hospitals_timedist_filled.parquet",
    "usa_low_detail_county": processed_dir / "usa_low_detail_county_county_travel_time_matrix_filled.parquet",
}

tract_lookup = (
    tracts[["GEOID", "NAMELSAD", "STATEFP", "COUNTYFP"]]
    .drop_duplicates("GEOID")
    .assign(
        origin_name=lambda df: np.where(
            df["GEOID"].astype(str) == "27013171600",
            "Mankato Census Tract 1716, Blue Earth, MN",
            df["NAMELSAD"],
        )
    )
    .rename(columns={"GEOID": "origin_id"})[["origin_id", "origin_name"]]
)

print(tract_lookup)

county_lookup = (
    tracts.assign(county_fips=tracts["STATEFP"].astype(str) + tracts["COUNTYFP"].astype(str))
    [["county_fips"]]
    .drop_duplicates("county_fips")
    .assign(origin_name=lambda df: "County FIPS " + df["county_fips"].astype(str))
    .rename(columns={"county_fips": "origin_id"})
)
destination_lookup = (
    facilities[["AHA_ID", "HOSPITAL_NAME"]]
    .drop_duplicates("AHA_ID")
    .assign(AHA_ID=lambda s: s["AHA_ID"].astype(str))
    .rename(columns={"AHA_ID": "destination_id", "HOSPITAL_NAME": "destination_name"})
)

loaded_matrices = {}
for profile_id, path in processed_paths.items():
    assert path.exists(), f"Processed matrix not found: {path}"
    df = pd.read_parquet(path).copy()
    df["origin_id"] = df["origin_id"].astype(str)
    df["destination_id"] = df["destination_id"].astype(str)

    origin_lookup = tract_lookup if profile_id == "mn_high_detail" else county_lookup
    df = df.merge(origin_lookup, on="origin_id", how="left")
    df = df.merge(destination_lookup, on="destination_id", how="left")
    loaded_matrices[profile_id] = df

    raw_inf = int((~np.isfinite(df["duration_min_raw"])).sum())
    filled_inf = int((~np.isfinite(df["duration_min_filled"])).sum())
    print(f"{profile_id}:")
    print(f"  Path      : {path}")
    print(f"  Rows      : {len(df):,}")
    print(f"  Raw inf   : {raw_inf:,}")
    print(f"  Filled inf: {filled_inf:,}")
    print()

preview_cols = [
    "origin_id",
    "origin_name",
    "destination_id",
    "destination_name",
    "duration_min_raw",
    "duration_min_filled",
    "duration_fill_method",
]
for profile_id, df in loaded_matrices.items():
    print(f"Preview — {profile_id}")
    display(df[preview_cols].head(10))

         origin_id           origin_name
0      01101001800       Census Tract 18
1      01101001900       Census Tract 19
2      01101002000       Census Tract 20
3      01101002100       Census Tract 21
4      01001020700      Census Tract 207
...            ...                   ...
84410  56035000103     Census Tract 1.03
84411  56035000104     Census Tract 1.04
84412  56037970905  Census Tract 9709.05
84413  56037970904  Census Tract 9709.04
84414  56021001100       Census Tract 11

[84415 rows x 2 columns]
mn_high_detail:
  Path      : C:\Users\tngzj\OneDrive\Desktop\Heatmap_Hackathon\Data\output\Travel Dist Processed\valhalla_mn_hospitals_timedist_filled.parquet
  Rows      : 12,905
  Raw inf   : 111
  Filled inf: 0

usa_low_detail_county:
  Path      : C:\Users\tngzj\OneDrive\Desktop\Heatmap_Hackathon\Data\output\Travel Dist Processed\usa_low_detail_county_county_travel_time_matrix_filled.parquet
  Rows      : 23,572
  Raw inf   : 273
  Filled inf: 0

Preview — mn_high_detail


,origin_id,origin_name,destination_id,destination_name,duration_min_raw,duration_min_filled,duration_fill_method
0,27137002000,Census Tract 20,6610410,Essentia Health St. Mary's Medical Center,4.100000,4.100000,raw
1,27137002000,Census Tract 20,6610390,Miller-Dwan Burn Center at Essentia Health Duluth,3.583333,3.583333,raw
2,27137002000,Census Tract 20,6610400,St. Luke's Hospital,5.450000,5.450000,raw
3,27137002000,Census Tract 20,6610085,Mercy Hospital,137.683333,137.683333,raw
4,27137002000,Census Tract 20,6611540,CentraCare - St. Cloud Hospital,153.250000,153.250000,raw
5,27137013400,Census Tract 134,6610400,St. Luke's Hospital,62.900000,62.900000,raw
6,27137013400,Census Tract 134,6610410,Essentia Health St. Mary's Medical Center,61.800000,61.800000,raw
7,27137013400,Census Tract 134,6610390,Miller-Dwan Burn Center at Essentia Health Duluth,62.300000,62.300000,raw
8,27137013400,Census Tract 134,6611540,CentraCare - St. Cloud Hospital,189.866667,189.866667,raw
9,27137013400,Census Tract 134,6610085,Mercy Hospital,174.300000,174.300000,raw


Preview — usa_low_detail_county


,origin_id,origin_name,destination_id,destination_name,duration_min_raw,duration_min_filled,duration_fill_method
0,01001,County FIPS 01001,6530304,University of Alabama at Birmingham Hospital (...,78.800000,78.800000,raw
1,01001,County FIPS 01001,6530170,Children's of Alabama (Children's of Alabama B...,78.083333,78.083333,raw
2,01001,County FIPS 01001,6530705,Baptist Medical Center South,30.716667,30.716667,raw
3,01001,County FIPS 01001,6380430,Piedmont Columbus Regional Midtown,108.050000,108.050000,raw
4,01003,County FIPS 01003,6530600,USA Health University Hospital (Arnold Luterma...,43.800000,43.800000,raw
5,01003,County FIPS 01003,6390792,Baptist Hospital,53.400000,53.400000,raw
6,01003,County FIPS 01003,6390830,Ascension Sacred Heart Pensacola,49.666667,49.666667,raw
7,01003,County FIPS 01003,6390253,Fort Walton Beach Medical Center,102.616667,102.616667,raw
8,01003,County FIPS 01003,6540300,Memorial Hospital at Gulfport,103.016667,103.016667,raw
9,01005,County FIPS 01005,6530373,Southeast Alabama Medical Center,72.316667,72.316667,raw
